# N057 · 二叉堆、建堆与优先队列

**目标：** 实现堆不变量并区分建堆与逐个插入成本。

**先修：** N040, N009, N014。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 完全二叉树；sift_up/down；heapify；最小/最大堆；稳定次序键。

## 从问题到算法

完全二叉树按数组顺序存储，i的孩子是2i+1与2i+2。插入只可能破坏祖先链，删除堆顶只可能破坏向下路径。自底向上建堆比逐个插入更便宜，因为多数节点靠近叶子，实际下沉距离很短。

## 最小实现

**本章接口：** `MinHeap`、`heapify_bottom_up(a)`、`last_stone_weight(stones)`

In [1]:
def heapify_bottom_up(a):
    def sift(i):
        while 2*i+1<len(a):
            child=2*i+1
            if child+1<len(a) and a[child+1]<a[child]: child+=1
            if a[i]<=a[child]: break
            a[i],a[child]=a[child],a[i]; i=child
    for i in range(len(a)//2-1,-1,-1): sift(i)
    return a

class MinHeap:
    def __init__(self,values=()): self.data=heapify_bottom_up(list(values))
    def push(self,x):
        self.data.append(x); i=len(self.data)-1
        while i and self.data[i]<self.data[(i-1)//2]:
            p=(i-1)//2; self.data[i],self.data[p]=self.data[p],self.data[i]; i=p
    def pop(self):
        if not self.data: raise IndexError('empty heap')
        answer=self.data[0]; last=self.data.pop()
        if self.data:
            self.data[0]=last; i=0
            while 2*i+1<len(self.data):
                c=2*i+1
                if c+1<len(self.data) and self.data[c+1]<self.data[c]: c+=1
                if self.data[i]<=self.data[c]: break
                self.data[i],self.data[c]=self.data[c],self.data[i]; i=c
        return answer

def last_stone_weight(stones):
    heap=MinHeap(-x for x in stones)
    while len(heap.data)>1:
        a=-heap.pop(); b=-heap.pop()
        if a!=b: heap.push(-(a-b))
    return -heap.pop() if heap.data else 0

## 正确性、前提与复杂度

子堆先合法，sift选择较小孩子交换，只需继续修复被替换的子树。高度h的节点数量约n/2^(h+1)，总下沉工作Σh·n/2^(h+1)=O(n)。

**代价：** 建堆O(n)，push/pop最坏O(log n)，空间O(n)；石头模拟O(n log n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

h=MinHeap(); rows=[]
for x in [5,2,7,1,3]:
    h.push(x); rows.append((x,h.data[:]))
show_table(['插入值','堆数组'],rows)
show_table(['数组索引','父索引','值'],[(i,(i-1)//2 if i else None,v) for i,v in enumerate(h.data)])

插入值,堆数组
5,[5]
2,"[2, 5]"
7,"[2, 5, 7]"
1,"[1, 2, 7, 5]"
3,"[1, 2, 7, 5, 3]"


数组索引,父索引,值
0,None,1
1,0,2
2,0,7
3,1,5
4,1,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import heapq
from random import Random
rng=Random(57); h=MinHeap(); ref=[]
for _ in range(1000):
    if not ref or rng.randrange(3):
        x=rng.randrange(-50,51); h.push(x); heapq.heappush(ref,x)
    else: assert h.pop()==heapq.heappop(ref)
    assert all(h.data[(i-1)//2]<=h.data[i] for i in range(1,len(h.data)))
assert [h.pop() for _ in range(len(h.data))]==sorted(ref)
assert last_stone_weight([2,7,4,1,8,1])==1
assert last_stone_weight([3])==3
print('N057: 所有本章断言通过')

N057: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明自底向上建堆的线性成本。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用负号封装最大堆并说明数据类型限制。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1046. Last Stone Weight（canonical）
- 703. Kth Largest Element in a Stream（preview）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。